# 1. Loading the Datasets

In [ ]:
import sys
sys.path.append("..")
import helpers as h

## 1.1 Load both datasets from scikit-learn

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
data = load_wine()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

We import the libraries and load the Wine dataset from scikit-learn. `X` is a DataFrame with the original feature names and `y` is a Series with the class labels. The target is not inside `X`.

## 1.2 Create the data frames

In [3]:
print(X.head())
print(X.shape)
print(list(X.columns))
print(sorted(y.unique()), list(data.target_names))
print(y.value_counts().sort_index())
print(dict(enumerate(data.target_names)))     # label -> class name mapping

   alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
0    14.23        1.71  2.43               15.6      127.0           2.80   
1    13.20        1.78  2.14               11.2      100.0           2.65   
2    13.16        2.36  2.67               18.6      101.0           2.80   
3    14.37        1.95  2.50               16.8      113.0           3.85   
4    13.24        2.59  2.87               21.0      118.0           2.80   

   flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
0        3.06                  0.28             2.29             5.64  1.04   
1        2.76                  0.26             1.28             4.38  1.05   
2        3.24                  0.30             2.81             5.68  1.03   
3        3.49                  0.24             2.18             7.80  0.86   
4        2.69                  0.39             1.82             4.32  1.04   

   od280/od315_of_diluted_wines  proline  
0                  

Here we look at the first 5 rows, the shape, the column names, the labels and the class names.

- The dataset has 178 samples and 13 numerical features (chemical measurements of wines such as alcohol, flavanoids, color_intensity and proline).
- The labels are 0, 1 and 2. scikit-learn names them `class_0`, `class_1` and `class_2` (three different grape cultivars), so the mapping is {0: class_0, 1: class_1, 2: class_2}.
- The class counts are 59, 71 and 48.

# 2. Dataset Quality Checks

## 2.1 Missing values and duplicates

In [4]:
print(X.isnull().sum())            # per-column missing counts
print("total missing:", X.isnull().sum().sum(), "| duplicate rows:", X.duplicated().sum())

alcohol                         0
malic_acid                      0
ash                             0
alcalinity_of_ash               0
magnesium                       0
total_phenols                   0
flavanoids                      0
nonflavanoid_phenols            0
proanthocyanins                 0
color_intensity                 0
hue                             0
od280/od315_of_diluted_wines    0
proline                         0
dtype: int64
total missing: 0 | duplicate rows: 0


We check the missing values for every column and the duplicate rows. There are 0 missing values in all 13 columns and 0 duplicate rows, so no cleaning is needed.

## 2.2 Potential outliers

In [5]:
q1 = X.quantile(0.25)
q3 = X.quantile(0.75)
iqr = q3 - q1
outliers = (X < q1 - 1.5 * iqr) | (X > q3 + 1.5 * iqr)     # only counted, nothing is removed
outliers.sum().sort_values(ascending=False)

magnesium                       4
alcalinity_of_ash               4
color_intensity                 4
malic_acid                      3
ash                             3
proanthocyanins                 2
hue                             1
alcohol                         0
total_phenols                   0
nonflavanoid_phenols            0
flavanoids                      0
od280/od315_of_diluted_wines    0
proline                         0
dtype: int64

We use the IQR rule: a value is a potential outlier if it is below Q1 - 1.5 x IQR or above Q3 + 1.5 x IQR. We only **count** these values, we do not remove or change them.

- 7 of the 13 features have some potential outliers, but the numbers are small: magnesium, alcalinity_of_ash and color_intensity have 4 each, malic_acid and ash have 3, proanthocyanins has 2 and hue has 1. The other 6 features (including proline) have none.
- At most 4 of 178 samples (about 2%) are flagged in any feature, so the dataset quality is good.
- These values may still matter: KNN and K-Means use distances, so extreme values can affect them, and the mean and standard deviation used by StandardScaler are also influenced by outliers. Decision Trees and GaussianNB are less sensitive.

## 2.3 Data types

In [6]:
print(X.dtypes)
print("numerical:", X.select_dtypes(include="number").shape[1],
      "categorical:", X.select_dtypes(exclude="number").shape[1])

alcohol                         float64
malic_acid                      float64
ash                             float64
alcalinity_of_ash               float64
magnesium                       float64
total_phenols                   float64
flavanoids                      float64
nonflavanoid_phenols            float64
proanthocyanins                 float64
color_intensity                 float64
hue                             float64
od280/od315_of_diluted_wines    float64
proline                         float64
dtype: object
numerical: 13 categorical: 0


All 13 features are `float64`: 13 numerical features and 0 categorical features. No encoding is needed.

# 3. Exploratory Data Analysis (EDA)

## 3.1 Descriptive statistics

In [7]:
stats = X.agg(["mean", "median", "min", "max", "std"]).T
stats["Q1"] = X.quantile(0.25)
stats["Q3"] = X.quantile(0.75)
stats

,mean,median,min,max,std,Q1,Q3
alcohol,13.000618,13.050,11.03,14.83,0.811827,12.3625,13.6775
malic_acid,2.336348,1.865,0.74,5.80,1.117146,1.6025,3.0825
ash,2.366517,2.360,1.36,3.23,0.274344,2.2100,2.5575
alcalinity_of_ash,19.494944,19.500,10.60,30.00,3.339564,17.2000,21.5000
magnesium,99.741573,98.000,70.00,162.00,14.282484,88.0000,107.0000
total_phenols,2.295112,2.355,0.98,3.88,0.625851,1.7425,2.8000
flavanoids,2.029270,2.135,0.34,5.08,0.998859,1.2050,2.8750
nonflavanoid_phenols,0.361854,0.340,0.13,0.66,0.124453,0.2700,0.4375
proanthocyanins,1.590899,1.555,0.41,3.58,0.572359,1.2500,1.9500
color_intensity,5.058090,4.690,1.28,13.00,2.318286,3.2200,6.2000


The table shows mean, median, min, max, standard deviation, Q1 and Q3 for every feature.

- The features are on very different scales. For example, proline has mean 746.9, range 278 - 1680 and standard deviation 314.9, while hue has mean 0.96, range 0.48 - 1.71 and standard deviation 0.23. If we gave the raw values to a distance-based model, proline would dominate the distance and hue would hardly matter. This is the reason for scaling later.
- For some features the mean and median are different (proline 746.9 vs 673.5, malic_acid 2.34 vs 1.87), which shows right-skewed distributions.

## 3.2 Class distribution and correlations

In [8]:
import matplotlib.pyplot as plt
import seaborn as sns

y.value_counts().sort_index().plot(kind="bar")
plt.title("Wine - class distribution")
plt.xlabel("Class (0, 1, 2)")
plt.ylabel("Number of samples")
plt.savefig("../figures/wine_class_distribution.png", dpi=150)
plt.show()

plt.figure(figsize=(11, 9))
sns.heatmap(X.corr(), cmap="coolwarm", annot=True, fmt=".2f", annot_kws={"size": 7})
plt.title("Wine - Pearson correlation")
plt.savefig("../figures/wine_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

corr = X.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))   # keep each pair once (matrix is symmetric)
upper.stack().sort_values(ascending=False).head(3)

total_phenols  flavanoids                      0.864564
flavanoids     od280/od315_of_diluted_wines    0.787194
total_phenols  od280/od315_of_diluted_wines    0.699949
dtype: float64

We create two charts and print the 3 strongest correlated pairs.

- **Class distribution:** the classes have 59, 71 and 48 samples. They are not perfectly equal but there is no extreme imbalance.
- **Correlation heatmap:** warm colors mean positive correlation, cool colors mean negative correlation and white means almost no linear relation. Some features are strongly related, some are not.
- **Top 3 pairs (absolute Pearson):** total_phenols - flavanoids (0.865), flavanoids - od280/od315 (0.787) and total_phenols - od280/od315 (0.700). Flavanoids are a kind of phenol, so it makes sense that they move together.
- All correlations are below 0.87, so no pair is almost a duplicate. We do not remove any feature because the assignment does not ask for feature selection.

## 3.3 Boxplots

In [9]:
h.draw_boxplots(X, 10, "../figures/wine", "Wine")

Every feature has its own small boxplot (2 figures), because the scales are very different.

- The orange line is the median, the box goes from Q1 (25%) to Q3 (75%), so the box contains the middle 50% of the data and its height shows the spread.
- The whiskers go up to 1.5 x IQR from the box. Points outside the whiskers are potential outliers.

Observations:
- Magnesium, alcalinity_of_ash and color_intensity show the most points beyond the whiskers (4 each), which matches the outlier table in 2.2.
- Proline has a very wide box (Q1 500.5, Q3 985) but no outliers, while hue has a narrow box and one high point (1.71).

# 4. Splitting the Datasets

## 4.1 Fix the test set and repeat the training/validation split

In [10]:
splits, X_test, y_test = h.make_splits(X, y)

We split the dataset once with `random_state=42` and `stratify=y` to get the fixed test set (20%, 36 samples). The test set is the same for all three repetitions and it is never used for training, scaling or model selection.

The remaining 80% is split again for each seed (42, 52, 62) with `test_size=0.125`, because 0.125 x 80% = 10% of all data. This gives 70% train (124), 10% validation (18) and 20% test (36).

Wine has 3 classes with different sizes (33%, 40%, 27%), so we use `stratify=y` to keep the same class proportions in every subset.

## 4.2 Verify the split

In [11]:
h.check_splits(splits, X_test, X)
h.split_summary(splits, y_test, y)

seed 42: no overlap = True, sizes add up = True, same test set = True
seed 52: no overlap = True, sizes add up = True, same test set = True
seed 62: no overlap = True, sizes add up = True, same test set = True


,seed,subset,n,percent,class_0,class_1,class_2
0,42,train,124,69.7,41,50,33
1,42,val,18,10.1,6,7,5
2,42,test,36,20.2,12,14,10
3,52,train,124,69.7,41,50,33
4,52,val,18,10.1,6,7,5
5,52,test,36,20.2,12,14,10
6,62,train,124,69.7,41,50,33
7,62,val,18,10.1,6,7,5
8,62,test,36,20.2,12,14,10


We have 3 checks for every seed:
- **Overlap:** no row appears in more than one subset (otherwise there would be data leakage).
- **Size:** train + validation + test must add up to 178 rows, so no row is lost or duplicated.
- **Test set:** everything that is not in train or validation must be exactly the test set, so the repetitions never touch the test data.

All checks are `True` for all three seeds. The table has 9 rows (3 seeds x train/validation/test) with the sizes, the percentages and the class counts. The class proportions are almost the same in every subset and the test rows are identical for every seed.

Limitation: the validation set has only 18 samples (6, 7 and 5 per class), so one wrong prediction changes the accuracy by about 5.6%. The three validation sets come from the same 80% pool, so they overlap and are not independent.

# 5. Data Preprocessing and Scaling

## 5.1 Handle missing values, if present

There are no missing values (verified in 2.1: 0 missing values in every column), so no imputer is needed.

## 5.2 Select one scaler

We choose **StandardScaler** (the assignment recommends it) and use the same scaler type for both datasets and all seeds. It subtracts the training mean and divides by the training standard deviation.

Why: the features have very different ranges (proline in the hundreds, hue around 1) and KNN and K-Means are distance-based, so large-range features would dominate. Compared with MinMaxScaler, StandardScaler does not squeeze most values into a narrow range when there are outliers.

The scaler was chosen before the experiments and we did not compare scalers or choose it by looking at validation or test results. A different scaler could give different KNN and K-Means results.

## 5.3 Fit and apply the scaler correctly

In [12]:
from sklearn.preprocessing import StandardScaler

def scale_df(scaler, df):
    return pd.DataFrame(scaler.transform(df), columns=df.columns, index=df.index)

scalers = {}
for seed in h.SEEDS:
    split = splits[seed]
    scaler = StandardScaler().fit(split["X_train"])      # fit on this seed's training data only, so nothing leaks from val/test
    split["X_train_scaled"] = scale_df(scaler, split["X_train"])
    split["X_val_scaled"] = scale_df(scaler, split["X_val"])
    scalers[seed] = scaler

X_test_scaled = scale_df(scalers[42], X_test)            # test uses only the seed-42 scaler
print(splits[42]["X_train"].head())
print(splits[42]["X_train_scaled"].head())

     alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
163    12.96        3.45  2.35               18.5      106.0           1.39   
148    13.32        3.24  2.38               21.5       92.0           1.93   
116    11.82        1.47  1.99               20.8       86.0           1.98   
122    12.42        4.43  2.73               26.5      102.0           2.20   
115    11.03        1.51  2.20               21.5       85.0           2.46   

     flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
163        0.70                  0.40             0.94             5.28  0.68   
148        0.76                  0.45             1.25             8.42  0.55   
116        1.60                  0.30             1.53             1.95  0.95   
122        2.13                  0.43             1.71             2.08  0.92   
115        2.17                  0.52             2.01             1.90  1.71   

     od280/od315_of_diluted_wines  pro

For each seed a new `StandardScaler` is fitted on that seed's training set only. The same scaler then transforms that seed's training and validation features. The test set is transformed once, with the seed-42 scaler, because the final evaluation uses the seed-42 model. If we fitted the scaler on all data, the mean and standard deviation of validation and test samples would leak into the training features.

Before and after (seed 42, first rows): the row indices stay the same, but the units are gone. For example row 163 has proline 675.0, which becomes -0.21 (a bit below the training mean), and row 115 has hue 1.71, which becomes +3.41, so it is more than 3 standard deviations above the mean (this is the hue outlier found in 2.2). Training features have mean 0 and standard deviation 1 by construction; validation and test features do not, because they use the training statistics.

# 6. Preparing the Two Data Representations

In [13]:
for seed in h.SEEDS:
    split = splits[seed]
    print(seed,
          "train:", split["X_train"].shape, split["X_train_scaled"].shape,
          "| same train rows:", split["X_train"].index.equals(split["X_train_scaled"].index),
          "| same val rows:", split["X_val"].index.equals(split["X_val_scaled"].index))

42 train: (124, 13) (124, 13) | same train rows: True | same val rows: True
52 train: (124, 13) (124, 13) | same train rows: True | same val rows: True
62 train: (124, 13) (124, 13) | same train rows: True | same val rows: True


We have two representations: **unscaled** (original values) and **scaled** (StandardScaler). For every seed the shapes are the same (for example seed 42: (124, 13) and (124, 13)) and the row indices are identical (`same train rows: True`, `same val rows: True`), so the same samples are used in both representations. Only the scaling is different.

# 7. Training the Classification Models

In [14]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report)

DATASET = "wine"

# Source: assignment 8.1 (macro metrics, zero_division=0, ROC-AUC rules)
def compute_metrics(model, X, y):
    pred = model.predict(X)
    proba = model.predict_proba(X)
    classes = list(model.classes_)                       # column order of predict_proba

    # Wine has 3 classes: one-vs-rest ROC-AUC for each class, then the macro average
    auc = roc_auc_score(y, proba, multi_class="ovr", average="macro", labels=classes)

    undefined = [f"{c}:precision" for c in classes if (pred == c).sum() == 0]
    undefined += [f"{c}:recall" for c in classes if (y == c).sum() == 0]
    return {
        "accuracy": accuracy_score(y, pred),
        "macro_precision": precision_score(y, pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y, pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
        "roc_auc": auc,
        "undefined_metrics": ";".join(undefined) if undefined else "none",
    }

# Fits ONE model and returns ONE result row
def run_one_model(model, X_train, y_train, X_val, y_val):
    model.fit(X_train, y_train)                    # train on the training data only
    train_pred = model.predict(X_train)            # training predictions, to see overfitting
    val = compute_metrics(model, X_val, y_val)     # validation metrics
    return {
        "train_accuracy": accuracy_score(y_train, train_pred),
        "train_macro_f1": f1_score(y_train, train_pred, average="macro", zero_division=0),
        "val_accuracy": val["accuracy"],
        "val_macro_precision": val["macro_precision"],
        "val_macro_recall": val["macro_recall"],
        "val_macro_f1": val["macro_f1"],
        "val_roc_auc": val["roc_auc"],
        "undefined_metrics": val["undefined_metrics"],
    }

## 7.1 Required experiments

In [15]:
pd.DataFrame({"algorithm": ["Decision Tree", "KNN", "GaussianNB"],
              "settings": ["max_depth = 2, 3, 5, 10, None", "n_neighbors = 1, 3, 5, 7, 11", "defaults"],
              "fits per dataset": [30, 30, 6]})

,algorithm,settings,fits per dataset
0,Decision Tree,"max_depth = 2, 3, 5, 10, None",30
1,KNN,"n_neighbors = 1, 3, 5, 7, 11",30
2,GaussianNB,defaults,6


Each dataset uses 2 representations (unscaled/scaled), 3 algorithms with the prescribed settings (Decision Tree 5 depths, KNN 5 values of k, GaussianNB with defaults) and 3 seeds. This gives 3 x 2 x 11 = 66 fits per dataset. Every model is fitted on the training data only.

## 7.2 Record and compare the experiments

In [16]:
# Runs the 22 configurations (2 representations x 11 settings) for ONE seed
def run_one_seed(seed, split):
    rows = []
    models = {}
    for representation in ["unscaled", "scaled"]:
        if representation == "scaled":
            X_train, X_val = split["X_train_scaled"], split["X_val_scaled"]
        else:
            X_train, X_val = split["X_train"], split["X_val"]
        for algorithm, parameter, model in h.get_configs(seed):
            result = run_one_model(model, X_train, split["y_train"], X_val, split["y_val"])
            run_id = f"{DATASET}_{algorithm}_{representation}_{parameter}_s{seed}"   # assignment 7.2
            rows.append({"dataset": DATASET, "run_id": run_id, "seed": seed,
                         "algorithm": algorithm, "representation": representation,
                         "parameter": parameter, **result})
            models[run_id] = model
    return rows, models


all_rows = []
models42 = {}
for seed in h.SEEDS:
    rows, models = run_one_seed(seed, splits[seed])
    all_rows += rows
    if seed == 42:
        models42 = models
runs = pd.DataFrame(all_rows)
runs

,dataset,run_id,seed,algorithm,representation,parameter,train_accuracy,train_macro_f1,val_accuracy,val_macro_precision,val_macro_recall,val_macro_f1,val_roc_auc,undefined_metrics
0,wine,wine_dt_unscaled_depth2_s42,42,dt,unscaled,depth2,0.935484,0.929694,0.833333,0.863095,0.811111,0.817521,0.910596,none
1,wine,wine_dt_unscaled_depth3_s42,42,dt,unscaled,depth3,0.991935,0.992584,0.944444,0.958333,0.944444,0.947475,0.968645,none
2,wine,wine_dt_unscaled_depth5_s42,42,dt,unscaled,depth5,1.000000,1.000000,0.944444,0.958333,0.944444,0.947475,0.957071,none
3,wine,wine_dt_unscaled_depth10_s42,42,dt,unscaled,depth10,1.000000,1.000000,0.944444,0.958333,0.944444,0.947475,0.957071,none
4,wine,wine_dt_unscaled_depthNone_s42,42,dt,unscaled,depthNone,1.000000,1.000000,0.944444,0.958333,0.944444,0.947475,0.957071,none
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,wine,wine_knn_scaled_k3_s62,62,knn,scaled,k3,0.959677,0.960888,0.888889,0.896825,0.904762,0.888500,1.000000,none
62,wine,wine_knn_scaled_k5_s62,62,knn,scaled,k5,0.975806,0.975871,0.944444,0.944444,0.952381,0.944056,1.000000,none
63,wine,wine_knn_scaled_k7_s62,62,knn,scaled,k7,0.967742,0.968371,0.944444,0.944444,0.952381,0.944056,1.000000,none
64,wine,wine_knn_scaled_k11_s62,62,knn,scaled,k11,0.967742,0.968371,0.944444,0.944444,0.952381,0.944056,1.000000,none


The table has all 66 runs with their training and validation metrics. Every run has a unique ID: `dataset_algorithm_representation_parameter_seed` (for example `wine_knn_scaled_k11_s42`). The seed-42 models are stored in `models42`, so the final test does not need any refitting.

In [17]:
h.get_versions()

{'python': '3.12.3',
 'numpy': '2.5.4',
 'pandas': '3.0.6',
 'scikit-learn': '1.9.1'}

These are the Python and package versions used in this run. They are also written in the README for reproducibility.

# 8. Measuring Validation Performance

## 8.1 Required metrics

In [18]:
runs["undefined_metrics"].value_counts()

undefined_metrics
none    66
Name: count, dtype: int64

We use the same metric conventions for all runs:
- accuracy, and **macro** precision, recall and F1 (the average of the 3 class scores with equal weight, so small classes count as much as big classes). `zero_division=0` is used when a denominator is zero.
- ROC-AUC for Wine is **one-vs-rest** for each class, averaged with the macro average, using the probability column of each class from `predict_proba`.

The `undefined_metrics` column records classes with undefined precision or recall before `zero_division=0` could hide them. In all 66 runs the value is `none`, so there was no undefined metric.

## 8.2 Comparison tables and model selection

In [19]:
METRICS = ["train_accuracy", "train_macro_f1", "val_accuracy", "val_macro_precision",
           "val_macro_recall", "val_macro_f1", "val_roc_auc"]
group_cols = ["dataset", "algorithm", "representation", "parameter"]

summary = runs.groupby(group_cols)[METRICS].agg(["mean", "std"])      # std uses ddof=1 by default
summary.columns = [name + "_" + stat for name, stat in summary.columns]   # ("val_macro_f1", "mean") -> "val_macro_f1_mean"
summary = summary.reset_index()
summary.insert(0, "config_id", summary["dataset"] + "_" + summary["algorithm"] + "_"
               + summary["representation"] + "_" + summary["parameter"])
summary

,config_id,dataset,algorithm,representation,parameter,train_accuracy_mean,train_accuracy_std,train_macro_f1_mean,train_macro_f1_std,val_accuracy_mean,val_accuracy_std,val_macro_precision_mean,val_macro_precision_std,val_macro_recall_mean,val_macro_recall_std,val_macro_f1_mean,val_macro_f1_std,val_roc_auc_mean,val_roc_auc_std
0,wine_dt_scaled_depth10,wine,dt,scaled,depth10,1.000000,0.000000,1.000000,0.000000,0.944444,0.055556,0.961420,0.037133,0.940741,0.061195,0.946156,0.054515,0.955219,0.045735
1,wine_dt_scaled_depth2,wine,dt,scaled,depth2,0.935484,0.008065,0.936466,0.010609,0.851852,0.032075,0.872354,0.021400,0.848677,0.049496,0.848148,0.036475,0.904970,0.021898
2,wine_dt_scaled_depth3,wine,dt,scaled,depth3,0.991935,0.000000,0.992584,0.000000,0.944444,0.055556,0.961420,0.037133,0.940741,0.061195,0.946156,0.054515,0.961141,0.043103
3,wine_dt_scaled_depth5,wine,dt,scaled,depth5,1.000000,0.000000,1.000000,0.000000,0.944444,0.055556,0.961420,0.037133,0.940741,0.061195,0.946156,0.054515,0.955219,0.045735
4,wine_dt_scaled_depthNone,wine,dt,scaled,depthNone,1.000000,0.000000,1.000000,0.000000,0.944444,0.055556,0.961420,0.037133,0.940741,0.061195,0.946156,0.054515,0.955219,0.045735
5,wine_dt_unscaled_depth10,wine,dt,unscaled,depth10,1.000000,0.000000,1.000000,0.000000,0.944444,0.055556,0.961420,0.037133,0.940741,0.061195,0.946156,0.054515,0.955219,0.045735
6,wine_dt_unscaled_depth2,wine,dt,unscaled,depth2,0.935484,0.008065,0.936466,0.010609,0.851852,0.032075,0.872354,0.021400,0.848677,0.049496,0.848148,0.036475,0.904970,0.021898
7,wine_dt_unscaled_depth3,wine,dt,unscaled,depth3,0.991935,0.000000,0.992584,0.000000,0.944444,0.055556,0.961420,0.037133,0.940741,0.061195,0.946156,0.054515,0.961141,0.043103
8,wine_dt_unscaled_depth5,wine,dt,unscaled,depth5,1.000000,0.000000,1.000000,0.000000,0.944444,0.055556,0.961420,0.037133,0.940741,0.061195,0.946156,0.054515,0.955219,0.045735
9,wine_dt_unscaled_depthNone,wine,dt,unscaled,depthNone,1.000000,0.000000,1.000000,0.000000,0.944444,0.055556,0.961420,0.037133,0.940741,0.061195,0.946156,0.054515,0.955219,0.045735


The 66 runs are summarised into 22 configurations (2 representations x 11 settings). For every configuration we take the mean and the sample standard deviation over the 3 seeds. (The table is in alphabetical order, the ranking is in the next cell.)

Overfitting can already be seen here: `wine_knn_unscaled_k1` has a training macro-F1 of 1.0 but a validation score of only about 0.67, and Decision Trees with `max_depth` 5, 10 or None have a training macro-F1 of 1.0 but a validation score of about 0.95.

In [20]:
# Source: assignment 8.2 (selection rule, validation columns only)
ranked = summary.sort_values(
    by=["val_macro_f1_mean", "val_accuracy_mean", "val_macro_f1_std", "config_id"],
    ascending=[False, False, True, True])
selected = ranked.iloc[0]

print("Selected:", selected["config_id"])
ranked[["config_id", "val_macro_f1_mean", "val_accuracy_mean", "val_macro_f1_std"]].head(5)

Selected: wine_knn_scaled_k11


,config_id,val_macro_f1_mean,val_accuracy_mean,val_macro_f1_std
11,wine_knn_scaled_k11,0.981352,0.981481,0.032299
13,wine_knn_scaled_k5,0.981352,0.981481,0.032299
14,wine_knn_scaled_k7,0.981352,0.981481,0.032299
10,wine_knn_scaled_k1,0.965812,0.962963,0.029608
20,wine_nb_scaled_default,0.963844,0.962963,0.031359


The configurations are ranked with the assignment rule using validation metrics only (test set not touched): highest mean validation macro-F1, then highest mean validation accuracy, then lowest macro-F1 standard deviation, then alphabetical configuration ID.

`wine_knn_scaled_k5`, `k7` and `k11` have exactly the same mean macro-F1 (0.9814), mean accuracy (0.9815) and standard deviation (0.0323). The first three rules cannot separate them, so the last rule decides: `wine_knn_scaled_k11` comes first alphabetically (as text "k11" is before "k5" because "1" is before "5"). So **`wine_knn_scaled_k11` is selected because of the tie-break rule, not because k = 11 is really better than k = 5 or k = 7.**

In [21]:
h.plot_param_comparison(summary, DATASET, "dt",  "../figures/wine_tree_depth.png")
h.plot_param_comparison(summary, DATASET, "knn", "../figures/wine_knn_k.png")

**Decision Tree:** the training macro-F1 rises with depth (0.936 at depth 2, 0.993 at depth 3, 1.0 from depth 5). The validation score jumps from 0.848 (depth 2) to 0.946 at depth 3 and then stays flat, so deeper trees only overfit. The scaled and unscaled lines are exactly on top of each other, because tree splits depend only on the order of the feature values.

**KNN:** scaling is very important. With scaled data the validation macro-F1 is between 0.946 and 0.981, with unscaled data it is only between 0.60 and 0.67. In the unscaled data proline (range 278 - 1680) dominates the distance. For k = 1 the training score is 1.0 because every training point is its own nearest neighbour. GaussianNB gives the same validation score (0.964) for both representations.

Limitation: the validation sets are small (18 samples) and the error bars are about 0.03 - 0.06, so differences such as k = 5 vs k = 7 are one or two samples and should not be over-interpreted.

# 9. Evaluating the Selected Models on the Test Sets

## 9.1 Final performance metrics

In [22]:
names = list(data.target_names)
model = models42[selected["config_id"] + "_s42"]                      # seed-42 model only
X_final = X_test_scaled if selected["representation"] == "scaled" else X_test
pred = model.predict(X_final)
test_metrics = compute_metrics(model, X_final, y_test)

print(classification_report(y_test, pred, target_names=names, zero_division=0))
report_dict = classification_report(y_test, pred, target_names=names, zero_division=0, output_dict=True)

classifier_test = pd.DataFrame([{
    "dataset": DATASET, "selected_config": selected["config_id"], "eval_seed": 42,
    "val_macro_f1_mean": selected["val_macro_f1_mean"], "val_macro_f1_std": selected["val_macro_f1_std"],
    "test_accuracy": test_metrics["accuracy"], "test_macro_precision": test_metrics["macro_precision"],
    "test_macro_recall": test_metrics["macro_recall"], "test_macro_f1": test_metrics["macro_f1"],
    "test_roc_auc": test_metrics["roc_auc"]}])
classifier_test

              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        12
     class_1       1.00      1.00      1.00        14
     class_2       1.00      1.00      1.00        10

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36



,dataset,selected_config,eval_seed,val_macro_f1_mean,val_macro_f1_std,test_accuracy,test_macro_precision,test_macro_recall,test_macro_f1,test_roc_auc
0,wine,wine_knn_scaled_k11,42,0.981352,0.032299,1.0,1.0,1.0,1.0,1.0


The selected model `wine_knn_scaled_k11` (seed-42 model with seed-42 scaling) is evaluated once on the fixed test set (36 samples).

- All five metrics are 1.0 (accuracy, macro precision, macro recall, macro F1 and ROC-AUC) and precision, recall and F1 are 1.00 for every class (support 12, 14 and 10).
- Mean validation macro-F1 was 0.981 (std 0.032), so the test score is a little higher. We do not read this as the model being better on test: both sets are very small and the validation score is an average over 3 seeds, while the test score comes from one split. The selected model is not changed after seeing the test result.

## 9.2 Confusion matrices

In [23]:
cm = h.plot_confusion(y_test, pred, names, "Wine - Confusion Matrix", "../figures/wine_confusion_matrix.png")
print(cm)
h.manual_check(cm, 1, "class_1", report_dict)       # class_1 versus all other classes

[[12  0  0]
 [ 0 14  0]
 [ 0  0 10]]
class_1: TP=14, FP=0, FN=0, TN=22


,manual,classification_report
precision,1.0,1.0
recall,1.0,1.0
f1,1.0,1.0


The confusion matrix is 3 x 3 (rows = Actual, columns = Predicted) and only the diagonal has values: 12, 14 and 10. All off-diagonal cells are 0, so there is no confusion between the classes and no largest off-diagonal confusion to explain.

Manual check for `class_1` versus all other classes: TP = 14, FP = 0, FN = 0, TN = 22 (the 12 + 10 samples of the other classes). Precision = 14 / (14 + 0) = 1.0, recall = 14 / (14 + 0) = 1.0 and F1 = 1.0. These are the same as the `class_1` row of the classification report (not the macro average). No denominator is zero, so no metric is undefined.

## 9.3 ROC curves

In [24]:
class_aucs, macro_auc = h.plot_roc_wine(model, X_final, y_test, names, "../figures/wine_roc_curve.png")
print(class_aucs)
print("macro OvR AUC:", macro_auc)

{np.str_('class_0'): 1.0, np.str_('class_1'): 1.0, np.str_('class_2'): 1.0}
macro OvR AUC: 1.0


There is one one-vs-rest ROC curve for each class and all three have AUC = 1.0, so the macro OvR AUC is also 1.0. Each curve goes straight to the top-left corner: there is a threshold where every sample of the class is found without any false positive.

How the threshold works (in our own words): the model gives a probability and we decide the class with a threshold. If we lower the threshold, more samples are called positive, so we find more true positives (TPR goes up) but we also get more false positives (FPR goes up). If we raise it, we have fewer false positives but more false negatives. The ROC curve shows all these thresholds. It is only a description of the final result; we do not choose a new threshold or tune anything with the test set.

## 9.4 Error interpretation

In [25]:
h.list_errors(y_test, pred, names)

,actual,predicted


The table is empty: the selected classifier makes **no error on the test set**, so there is no misclassification to describe and we do not invent one.

One error-free result is not enough to say that the model generalises perfectly: the test set has only 36 samples and it is one single split. The validation results were not perfect (mean macro-F1 0.981 with std 0.032, so at least one seed had a mistake), so a different or larger test set could show errors.

Other limitations: the validation sets are very small (18 samples), the three validation sets overlap because they come from the same 80% pool, only one predeclared seed (42) is used for the final test, and the selection between k = 5, 7 and 11 was decided by the tie-break rule.

# 10. K-Means Clustering

## 10.1 Fit the clustering model

In [26]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

N_CLUSTERS = 3       # Wine has 3 classes

# Silhouette needs at least 2 clusters and fewer clusters than samples, otherwise NaN
def safe_silhouette(X_part, clusters):
    n_found = len(np.unique(clusters))
    if n_found < 2 or n_found >= len(X_part):
        return np.nan
    return silhouette_score(X_part, clusters)

kmeans_rows = []
kmeans42 = {}
for seed in h.SEEDS:
    split = splits[seed]
    for representation in ["unscaled", "scaled"]:
        if representation == "scaled":
            X_train, X_val = split["X_train_scaled"], split["X_val_scaled"]
        else:
            X_train, X_val = split["X_train"], split["X_val"]

        km = KMeans(n_clusters=N_CLUSTERS, random_state=seed, n_init=10)
        km.fit(X_train)                                  # no labels are passed: K-Means only sees the features
        val_clusters = km.predict(X_val)

        kmeans_rows.append({
            "dataset": DATASET,
            "run_id": f"{DATASET}_kmeans_{representation}_k{N_CLUSTERS}_s{seed}",
            "seed": seed, "representation": representation, "n_clusters": N_CLUSTERS,
            "train_inertia": km.inertia_,
            "val_ari": adjusted_rand_score(split["y_val"], val_clusters),
            "val_silhouette": safe_silhouette(X_val, val_clusters)})
        if seed == 42:
            kmeans42[representation] = km                # kept for the final test (10.2)

kmeans_runs = pd.DataFrame(kmeans_rows)                  # 6 rows
kmeans_summary = kmeans_runs.groupby(["dataset", "representation"]).agg(
    train_inertia_mean=("train_inertia", "mean"), train_inertia_std=("train_inertia", "std"),
    val_ari_mean=("val_ari", "mean"), val_ari_std=("val_ari", "std"),
    val_silhouette_mean=("val_silhouette", "mean"), val_silhouette_std=("val_silhouette", "std"),
    valid_silhouette_count=("val_silhouette", "count")).reset_index()    # 2 rows
kmeans_runs

,dataset,run_id,seed,representation,n_clusters,train_inertia,val_ari,val_silhouette
0,wine,wine_kmeans_unscaled_k3_s42,42,unscaled,3,1.350496e+06,0.282836,0.486994
1,wine,wine_kmeans_scaled_k3_s42,42,scaled,3,8.747391e+02,1.000000,0.251637
2,wine,wine_kmeans_unscaled_k3_s52,52,unscaled,3,1.303675e+06,0.400588,0.555887
3,wine,wine_kmeans_scaled_k3_s52,52,scaled,3,8.904355e+02,1.000000,0.301008
4,wine,wine_kmeans_unscaled_k3_s62,62,unscaled,3,1.390754e+06,0.400588,0.686671
5,wine,wine_kmeans_scaled_k3_s62,62,scaled,3,8.722466e+02,1.000000,0.272587


In [27]:
kmeans_summary

,dataset,representation,train_inertia_mean,train_inertia_std,val_ari_mean,val_ari_std,val_silhouette_mean,val_silhouette_std,valid_silhouette_count
0,wine,scaled,8.791404e+02,9.860948,1.000000,0.000000,0.275077,0.024780,3
1,wine,unscaled,1.348308e+06,43580.909759,0.361337,0.067984,0.576517,0.101425,3


K-Means uses 3 clusters (the same as the number of classes) and is fitted on the training features only. The labels are not given to the model, they are only used afterwards to measure the agreement with the classes. For every seed and representation we calculate the training inertia, the validation ARI and the validation silhouette (6 runs).

- **ARI (adjusted Rand index):** 1.0 means the clusters agree perfectly with the classes (cluster numbers do not matter). It is not an accuracy percentage. Scaled: 1.000 (std 0.000), unscaled: 0.361 (std 0.068). So scaling helps a lot.
- **Silhouette:** unscaled 0.577, scaled 0.275. Silhouette only measures how compact the clusters are in the used feature space. In the unscaled data proline dominates the distances, so the clusters look compact in that space, but they do not agree with the classes (low ARI). A higher silhouette is therefore not a better result here.
- **Inertia:** scaled about 879 and unscaled about 1.35 million. These cannot be compared because the units are different.
- The validation silhouette was valid in all 3 seeds for both representations (`valid_silhouette_count = 3`).

## 10.2 Final cluster evaluation on the fixed test sets

In [28]:
test_rows = []
test_clusters = {}
for representation in ["unscaled", "scaled"]:
    X_eval = X_test_scaled if representation == "scaled" else X_test
    clusters = kmeans42[representation].predict(X_eval)
    test_clusters[representation] = clusters

    counts = pd.Series(clusters).value_counts().sort_index()
    test_rows.append({
        "dataset": DATASET, "representation": representation, "eval_seed": 42,
        "n_test": len(y_test),
        "cluster_counts": ";".join(f"{c}:{n}" for c, n in counts.items()),
        "test_ari": adjusted_rand_score(y_test, clusters),
        "test_silhouette": safe_silhouette(X_eval, clusters)})

    print(representation)       # class-versus-cluster table
    print(pd.crosstab(y_test.map(dict(enumerate(names))).rename("class"),
                      pd.Series(clusters, index=y_test.index, name="cluster")))

kmeans_test = pd.DataFrame(test_rows)                    # 2 rows
kmeans_test

unscaled
cluster  0   1  2
class            
class_0  2  10  0
class_1  5   0  9
class_2  7   0  3
scaled
cluster   0   1   2
class              
class_0  12   0   0
class_1   1  13   0
class_2   0   0  10


,dataset,representation,eval_seed,n_test,cluster_counts,test_ari,test_silhouette
0,wine,unscaled,42,36,0:14;1:10;2:12,0.373323,0.49288
1,wine,scaled,42,36,0:13;1:13;2:10,0.908794,0.27112


The seed-42 K-Means models are applied to the fixed test set without refitting. The table shows the class-versus-cluster counts (cluster numbers are arbitrary).

- **Scaled:** test ARI = 0.909 and the clusters follow the classes closely: all 12 `class_0` samples are in cluster 0, all 10 `class_2` samples are in cluster 2 and 13 of the 14 `class_1` samples are in cluster 1 (1 sample is in cluster 0).
- **Unscaled:** test ARI = 0.373 and the classes are mixed: for example `class_1` is split into cluster 0 (5) and cluster 2 (9), and `class_2` into cluster 0 (7) and cluster 2 (3).
- Test silhouette is again higher for unscaled (0.493) than for scaled (0.271), for the same reason as in 10.1.

K-Means does not use labels, so its ARI cannot be compared with the classifier metrics: it solves a different task.

## 10.3 Visualise and interpret the clusters

In [29]:
corr_with_class = splits[42]["X_train"].corrwith(splits[42]["y_train"]).abs().sort_values(ascending=False)
feature1, feature2 = corr_with_class.index[:2]
print(feature1, feature2)
print(corr_with_class.head(4))

for representation in ["unscaled", "scaled"]:
    X_eval = X_test_scaled if representation == "scaled" else X_test
    h.plot_cluster_pair(X_eval, y_test, test_clusters[representation], feature1, feature2, names,
                        f"Wine - {representation}",
                        f"../figures/wine_kmeans_{representation}.png")

flavanoids od280/od315_of_diluted_wines
flavanoids                      0.871853
od280/od315_of_diluted_wines    0.775412
total_phenols                   0.743515
proline                         0.652525
dtype: float64


We plot the two features with the highest absolute correlation with the class in the seed-42 training data: `flavanoids` and `od280/od315_of_diluted_wines`. The left panel shows the true classes and the right panel shows the predicted clusters (colors of the two panels do not have to match, because cluster numbers are arbitrary).

- **Scaled:** the clusters follow the class groups: `class_2` is in the lower left, `class_1` is in the middle and `class_0` is in the upper right. The one visible mistake is a `class_1` point at high flavanoids that is put in the `class_0` cluster.
- **Unscaled:** the clusters are mixed in this view, they do not follow the class groups. This is because the clusters were found in all 13 features and mostly by proline, which is not one of the two plotted features.
- Limitation: this is only a 2D view of a 13-dimensional space, `class_0` and `class_1` overlap in these two features, and the colors are only valid inside each panel.

# 12. Export Results

In [30]:
h.save_part(runs, "classifier_runs", DATASET)
h.save_part(summary, "classifier_summary", DATASET)
h.save_part(classifier_test, "classifier_test", DATASET)
h.save_part(kmeans_runs, "kmeans_runs", DATASET)
h.save_part(kmeans_summary, "kmeans_summary", DATASET)
h.save_part(kmeans_test, "kmeans_test", DATASET)
h.combine_results()

classifier_runs: waiting for the other notebook
classifier_summary: waiting for the other notebook
classifier_test: waiting for the other notebook
kmeans_runs: waiting for the other notebook
kmeans_summary: waiting for the other notebook
kmeans_test: waiting for the other notebook
